<a href="https://colab.research.google.com/github/Karthik-velpula/MLOps/blob/main/Lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install feast==0.64.0 pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires tenacity<1

In [2]:
import feast

print("Feast version:", feast.__version__)

Feast version: 0.64.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
from google.colab import files

uploaded = files.upload()

Saving skill_gap_dataset_realistic.csv to skill_gap_dataset_realistic.csv


In [5]:
import pandas as pd

df = pd.read_csv("skill_gap_dataset_realistic.csv")

print(df.head())
print(df.shape)

  Skill_Name  Curriculum_Coverage  Industry_Demand  Job_Frequency  \
0     Python                    5                5             10   
1       Java                    5                4              9   
2        C++                    5                3              7   
3        SQL                    4                5              9   
4       DBMS                    5                4              8   

   Expert_Importance Gap_Status  
0                  5    Aligned  
1                  5    Aligned  
2                  4    Aligned  
3                  5    Aligned  
4                  5    Aligned  
(100, 6)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
print(df.columns)
print(df.info())
print(df.isnull().sum())

Index(['Skill_Name', 'Curriculum_Coverage', 'Industry_Demand', 'Job_Frequency',
       'Expert_Importance', 'Gap_Status'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Skill_Name           100 non-null    object
 1   Curriculum_Coverage  100 non-null    int64 
 2   Industry_Demand      100 non-null    int64 
 3   Job_Frequency        100 non-null    int64 
 4   Expert_Importance    100 non-null    int64 
 5   Gap_Status           100 non-null    object
dtypes: int64(4), object(2)
memory usage: 4.8+ KB
None
Skill_Name             0
Curriculum_Coverage    0
Industry_Demand        0
Job_Frequency          0
Expert_Importance      0
Gap_Status             0
dtype: int64


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
import pandas as pd

df = pd.read_csv("skill_gap_dataset_realistic.csv")

df["skill_name"] = df["Skill_Name"].astype(str)

df["Curriculum_Coverage"] = df["Curriculum_Coverage"].fillna(
    df["Curriculum_Coverage"].median()
)

df["Industry_Demand"] = df["Industry_Demand"].fillna(
    df["Industry_Demand"].median()
)

df["Job_Frequency"] = df["Job_Frequency"].fillna(
    df["Job_Frequency"].median()
)

df["Expert_Importance"] = df["Expert_Importance"].fillna(
    df["Expert_Importance"].median()
)

df["skill_encoded"] = df["Skill_Name"].astype("category").cat.codes

df["industry_gap"] = (
    df["Industry_Demand"] - df["Curriculum_Coverage"]
).astype("int64")

df["job_importance"] = (
    df["Job_Frequency"] * df["Expert_Importance"]
).astype("int64")

df["curriculum_coverage"] = df["Curriculum_Coverage"].astype("int64")
df["industry_demand"] = df["Industry_Demand"].astype("int64")
df["job_frequency"] = df["Job_Frequency"].astype("int64")
df["expert_importance"] = df["Expert_Importance"].astype("int64")

df["gap_status"] = df["Gap_Status"].astype(str)

print(df.head())
print(df.shape)

  Skill_Name  Curriculum_Coverage  Industry_Demand  Job_Frequency  \
0     Python                    5                5             10   
1       Java                    5                4              9   
2        C++                    5                3              7   
3        SQL                    4                5              9   
4       DBMS                    5                4              8   

   Expert_Importance Gap_Status skill_name  skill_encoded  industry_gap  \
0                  5    Aligned     Python             21             0   
1                  5    Aligned       Java             14            -1   
2                  4    Aligned        C++              2            -2   
3                  5    Aligned        SQL             23             1   
4                  5    Aligned       DBMS              7            -1   

   job_importance  curriculum_coverage  industry_demand  job_frequency  \
0              50                    5                5     

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [8]:
df[
    [
        "skill_name",
        "skill_encoded",
        "curriculum_coverage",
        "industry_demand",
        "job_frequency",
        "expert_importance",
        "industry_gap",
        "job_importance",
        "gap_status"
    ]
].head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,skill_name,skill_encoded,curriculum_coverage,industry_demand,job_frequency,expert_importance,industry_gap,job_importance,gap_status
0,Python,21,5,5,10,5,0,50,Aligned
1,Java,14,5,4,9,5,-1,45,Aligned
2,C++,2,5,3,7,4,-2,28,Aligned
3,SQL,23,4,5,9,5,1,45,Aligned
4,DBMS,7,5,4,8,5,-1,40,Aligned


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [9]:
base_time = pd.Timestamp(
    "2026-01-01",
    tz="UTC"
)

df["event_timestamp"] = (
    base_time +
    pd.to_timedelta(
        df.index,
        unit="s"
    )
)

df["created_timestamp"] = (
    df["event_timestamp"] +
    pd.Timedelta(seconds=1)
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [10]:
feature_df = df[
    [
        "skill_name",
        "event_timestamp",
        "created_timestamp",
        "skill_encoded",
        "curriculum_coverage",
        "industry_demand",
        "job_frequency",
        "expert_importance",
        "industry_gap",
        "job_importance"
    ]
].copy()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [11]:
label_df = df[
    [
        "skill_name",
        "event_timestamp",
        "gap_status"
    ]
].copy()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
print("Feature data:")
display(feature_df.head())

print("Labels:")
display(label_df.head())

Feature data:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,skill_name,event_timestamp,created_timestamp,skill_encoded,curriculum_coverage,industry_demand,job_frequency,expert_importance,industry_gap,job_importance
0,Python,2026-01-01 00:00:00+00:00,2026-01-01 00:00:01+00:00,21,5,5,10,5,0,50
1,Java,2026-01-01 00:00:01+00:00,2026-01-01 00:00:02+00:00,14,5,4,9,5,-1,45
2,C++,2026-01-01 00:00:02+00:00,2026-01-01 00:00:03+00:00,2,5,3,7,4,-2,28
3,SQL,2026-01-01 00:00:03+00:00,2026-01-01 00:00:04+00:00,23,4,5,9,5,1,45
4,DBMS,2026-01-01 00:00:04+00:00,2026-01-01 00:00:05+00:00,7,5,4,8,5,-1,40


Labels:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,skill_name,event_timestamp,gap_status
0,Python,2026-01-01 00:00:00+00:00,Aligned
1,Java,2026-01-01 00:00:01+00:00,Aligned
2,C++,2026-01-01 00:00:02+00:00,Aligned
3,SQL,2026-01-01 00:00:03+00:00,Aligned
4,DBMS,2026-01-01 00:00:04+00:00,Aligned


In [13]:
import os

repo_path = "/content/skill_gap_feast"

os.makedirs(
    f"{repo_path}/data",
    exist_ok=True
)

In [15]:
feature_df.to_parquet(
    f"{repo_path}/data/skill_gap_features.parquet",
    index=False
)

In [17]:
feature_store_yaml = """
project: skill_gap_project

registry: data/registry.db

provider: local

offline_store:
  type: file

online_store:
  type: sqlite
  path: data/online_store.db
"""

with open(
    f"{repo_path}/feature_store.yaml",
    "w"
) as f:
    f.write(feature_store_yaml)

In [18]:
feature_definition = '''
from datetime import timedelta

from feast import (
    Entity,
    FeatureView,
    FeatureService,
    Field,
    FileSource
)

from feast.types import (
    Float32,
    Int64
)

skill = Entity(
    name="skill",
    join_keys=["skill_name"],
    description="Skill in the skill gap dataset"
)

skill_gap_source = FileSource(
    name="skill_gap_source",
    path="data/skill_gap_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp"
)

skill_gap_feature_view = FeatureView(
    name="skill_gap_features",
    entities=[skill],

    ttl=timedelta(days=50000),

    schema=[
        Field(name="skill_encoded", dtype=Int64),
        Field(name="curriculum_coverage", dtype=Int64),
        Field(name="industry_demand", dtype=Int64),
        Field(name="job_frequency", dtype=Int64),
        Field(name="expert_importance", dtype=Int64),
        Field(name="industry_gap", dtype=Int64),
        Field(name="job_importance", dtype=Int64),
    ],

    source=skill_gap_source,

    online=True
)

skill_gap_feature_service = FeatureService(
    name="skill_gap_service",
    features=[
        skill_gap_feature_view
    ]
)
'''

with open(
    f"{repo_path}/features.py",
    "w"
) as f:
    f.write(feature_definition)

In [19]:
!find /content/titanic_feast -maxdepth 2 -type f

/content/titanic_feast/data/skill_gap_features.parquet
/content/titanic_feast/data/skill_gap_labels.parquet
/content/titanic_feast/features.py
/content/titanic_feast/feature_store.yaml


In [20]:
%cd /content/titanic_feast

/content/titanic_feast


In [21]:
!feast apply

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [22]:
!feast entities list

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [23]:
!feast feature-views list

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [24]:
from feast import FeatureStore

store = FeatureStore(
    repo_path="/content/titanic_feast"
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [25]:
feature_service = store.get_feature_service(
    "skill_gap_service"
)

In [26]:
entity_df = label_df.copy()

display(entity_df.head())

,skill_name,event_timestamp,gap_status
0,Python,2026-01-01 00:00:00+00:00,Aligned
1,Java,2026-01-01 00:00:01+00:00,Aligned
2,C++,2026-01-01 00:00:02+00:00,Aligned
3,SQL,2026-01-01 00:00:03+00:00,Aligned
4,DBMS,2026-01-01 00:00:04+00:00,Aligned


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [27]:
training_data = store.get_historical_features(
    entity_df=entity_df,
    features=feature_service
).to_df()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [28]:
display(training_data.head())

,skill_name,event_timestamp,gap_status,skill_encoded,curriculum_coverage,industry_demand,job_frequency,expert_importance,industry_gap,job_importance
0,Python,2026-01-01 00:00:00+00:00,Aligned,21,5,5,10,5,0,50
1,Java,2026-01-01 00:00:01+00:00,Aligned,14,5,4,9,5,-1,45
2,C++,2026-01-01 00:00:02+00:00,Aligned,2,5,3,7,4,-2,28
3,SQL,2026-01-01 00:00:03+00:00,Aligned,23,4,5,9,5,1,45
4,DBMS,2026-01-01 00:00:04+00:00,Aligned,7,5,4,8,5,-1,40


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [29]:
feature_columns = [
    "skill_encoded",
    "curriculum_coverage",
    "industry_demand",
    "job_frequency",
    "expert_importance",
    "industry_gap",
    "job_importance"
]

In [30]:
X = training_data[feature_columns]

y = training_data["gap_status"]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [32]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DecisionTreeClassifier(max_depth=4, random_state=42)

In [33]:
predictions = model.predict(X_test)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [34]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    predictions
)

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

Accuracy: 80.0 %


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [35]:
%cd /content/titanic_feast

!feast materialize \
    2026-01-01T00:00:00 \
    2026-01-02T00:00:00

/content/titanic_feast
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/c

In [37]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {
            "skill_name": "Python"
        }
    ]
).to_dict()

In [38]:
print(online_features)

{'skill_name': ['Python'], 'curriculum_coverage': [5], 'job_importance': [50], 'skill_encoded': [21], 'industry_demand': [4], 'industry_gap': [-1], 'expert_importance': [5], 'job_frequency': [10]}


In [39]:
online_df = pd.DataFrame(
    online_features
)

display(online_df)

,skill_name,curriculum_coverage,job_importance,skill_encoded,industry_demand,industry_gap,expert_importance,job_frequency
0,Python,5,50,21,4,-1,5,10


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [40]:
final_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

final_model.fit(X, y)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DecisionTreeClassifier(max_depth=4, random_state=42)

In [41]:
X_online = online_df[
    feature_columns
]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [42]:
prediction = final_model.predict(
    X_online
)

print(
    "Predicted gap status:",
    prediction[0]
)

Predicted gap status: Aligned


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [43]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {"skill_name": "Python"},
        {"skill_name": "Java"},
        {"skill_name": "C++"},
        {"skill_name": "SQL"}
    ]
).to_dict()

online_df = pd.DataFrame(
    online_features
)

display(online_df)

,skill_name,curriculum_coverage,job_importance,skill_encoded,industry_demand,industry_gap,expert_importance,job_frequency
0,Python,5,50,21,4,-1,5,10
1,Java,5,45,14,4,-1,5,9
2,C++,5,28,2,3,-2,4,7
3,SQL,4,45,23,5,1,5,9


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [44]:
online_df["predicted_gap_status"] = (
    final_model.predict(
        online_df[feature_columns]
    )
)

display(
    online_df[
        [
            "skill_name",
            "predicted_gap_status"
        ]
    ]
)

,skill_name,predicted_gap_status
0,Python,Aligned
1,Java,Aligned
2,C++,Aligned
3,SQL,Aligned


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
